# MGS-17 — Contrôle de paramètres (Parameter Control)

> **Capacite du moteur exercee :** faire varier les paramètres de l'algorithme *pendant* la course, plutot que de les fixer une fois pour toutes — et mesurer quand cela aide vraiment.

La [**selection d'algorithme** (MGS-16)](MGS-16-AlgorithmSelection.ipynb) etait une premiere réponse au **theoreme No-Free-Lunch** (Wolpert & Macready, 1997) : aucun optimiseur ne domine universellement, donc on *choisit* le bon selon le paysage. Il existe une **deuxieme réponse**, complementaire : pour un même optimiseur, **adapter ses paramètres au cours du temps**. C'est le **contrôle de paramètres** (Eiben, Hinterding & Michalewicz, 1999 ; Karafotias, Hoogendoorn & Eiben, 2015).

## Taxonomie d'Eiben : reglage vs contrôle

| Stratégie | Quand le paramètre est-il fixe ? | Exemple |
|-----------|----------------------------------|---------|
| **Reglage** (*tuning*, hors-ligne) | Avant la course, identique chaque generation | `mutationProbability = 0.2` |
| **Contrôle** (*control*, en ligne) | Il varie **pendant** la course | selon 3 sous-types ci-dessous |

Le contrôle se decompose en 3 sous-types, selon **ce qui pilote** la variation :

| Sous-type | Pilote | Ce notebook |
|-----------|--------|-------------|
| **Déterministe** | Une fonction du temps (numéro de generation) | §2 — schedule decroissant |
| **Adaptatif** | Le *feedback* de la population (diversite, succes) | §3 — diversite pilote la mutation |
| **Self-adaptatif** | L'individu lui-même (chaque chromosome a son propre taux) | §4 — taux heterogene par individu |

## Le fil rouge (spoiler honnete)

On pourrait croire que « contrôler c'est toujours mieux ». **Les données montreront le contraire.** Faire varier la mutation, c'est jouer sur le **compromis exploration/exploitation** : la mutation est le levier d'*exploration*. Sur un paysage **multimodal** (Rastrigin), explorer aide a s'extraire des optima locaux ; sur un paysage **unimodal** (Sphere), l'exploration ajoutee **disrupte** la convergence rapide. Aucune stratégie de contrôle ne dominera sur les **deux** paysages : c'est le **No-Free-Lunch au niveau du paramètre**, en echo direct de [MGS-16](MGS-16-AlgorithmSelection.ipynb).

### Plan
1. **Reglage** (reference) : balayage des taux fixes sur Rastrigin.
2. **Contrôle déterministe** : schedule `f(generation)` via `DynamicProbability`.
3. **Contrôle adaptatif** : la diversite de la population pilote le taux (esprit règle 1/5 de Rechenberg).
4. **Contrôle self-adaptatif** : un taux par individu via le contexte d'evolution.
5. Comparaison Rastrigin **vs** Sphere + taxonomie + exercices.

In [1]:
// Cablage : DLLs MetaGeneticSharp + GeneticSharp (build Debug net9.0).
// Pre-requis : dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;
using System;
using System.Linq;

Console.WriteLine("MetaGeneticSharp + GeneticSharp charges.");
Console.WriteLine("  ProbabilityConfig      : " + typeof(ProbabilityConfig).Name);
Console.WriteLine("  ProbabilityStrategy    : " + typeof(ProbabilityStrategy).Name);
Console.WriteLine("  OperatorsProbabilityConfig : " + typeof(OperatorsProbabilityConfig).Name);
Console.WriteLine("  RastriginFitness       : " + typeof(RastriginFitness).Name);
Console.WriteLine("  SphereFitness          : " + typeof(SphereFitness).Name);

MetaGeneticSharp + GeneticSharp charges.


  ProbabilityConfig      : ProbabilityConfig


  ProbabilityStrategy    : ProbabilityStrategy


  OperatorsProbabilityConfig : OperatorsProbabilityConfig


  RastriginFitness       : RastriginFitness


  SphereFitness          : SphereFitness


In [2]:
// DoubleArrayChromosome (replique fork, MGS-10/15/16) — genes reels.
public class DoubleArrayChromosome : ChromosomeBase
{
    private readonly double _min; private readonly double _max;
    public DoubleArrayChromosome(double[] values, double min, double max) : base(values.Length)
    { _min=min; _max=max; for (int i=0;i<values.Length;i++) ReplaceGene(i, new Gene(values[i])); }
    public override IChromosome CreateNew()
    { var rand=RandomizationProvider.Current; int n=Length; var v=new double[n];
      for (int i=0;i<n;i++) v[i]=rand.GetDouble(_min,_max); return new DoubleArrayChromosome(v,_min,_max); }
    public override Gene GenerateGene(int geneIndex) => new Gene(RandomizationProvider.Current.GetDouble(_min,_max));
    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

const int PopSize = 50;
const int NFE = 5000;   // budget d'evaluations -> ~100 generations

// MakeOptimizer : lance un GA avec la metaheuristique donnee (son ProbabilityConfig pilote la mutation).
// buildMh(gens) construit la metaheuristique ; renvoie la fitness maximisee.
Optimizer MakeOptimizer(Func<int,IMetaHeuristic> buildMh)
{ return (Optimizer)((req) => { int gens=Math.Max(1,req.Evaluations/PopSize);
    var (min,max)=req.Bounds; double mid=0.5*(min+max);
    var adam=new DoubleArrayChromosome(Enumerable.Repeat(mid,req.Dimension).ToArray(),min,max);
    var pop=new MetaPopulation(PopSize,PopSize,adam);
    var ga=new MetaGeneticAlgorithm(pop,req.Fitness,new EliteSelection(),new UniformCrossover(0.5f),new UniformMutation(true),buildMh(gens));
    ga.Termination=new GenerationNumberTermination(gens);
    ga.MutationProbability=0.2f;   // defaut : sera ECRASE si la metaheuristique utilise OverwriteProbability
    ga.Start();
    return ga.BestChromosome.Fitness ?? double.NegativeInfinity; }); }

// RunAvg : objectif moyen (proche 0 = meilleur) sur plusieurs seeds.
// Rastrigin est bruitee a NFE=5000 (cf MGS-10/16) : ne jamais juger sur 1 seed -> on moyenne sur 5.
double RunAvg(Func<int,IMetaHeuristic> build, OptimizerRequest req, int[] seeds)
{ double s=0;
  foreach (var sd in seeds) { FastRandomRandomization.ResetSeed(sd); s += -MakeOptimizer(build)(req); }
  return s/seeds.Length; }

var Seeds = new[]{7,42,99,123,777};
var reqR = new OptimizerRequest(new RastriginFitness(), KnownFunctionsBounds.For(typeof(RastriginFitness)), 5, NFE);
var reqS = new OptimizerRequest(new SphereFitness(),    KnownFunctionsBounds.For(typeof(SphereFitness)),    5, NFE);
Console.WriteLine($"Setup OK. PopSize={PopSize}, NFE={NFE} (gens={NFE/PopSize}), dim=5, seeds=[{string.Join(',',Seeds)}]");

Setup OK. PopSize=50, NFE=5000 (gens=100), dim=5, seeds=[7,42,99,123,777]


## 1. Reglage (tuning) — la reference fixe

Le **reglage** fixe le taux de mutation *avant* la course et ne le touche plus. C'est l'approche par defaut. On balaye `p in {0.01, 0.05, 0.2, 0.5}` sur Rastrigin pour etablir la **reference** : quel est le meilleur taux fixe, et surtout — un taux fixe est-il *robuste* ?

> **Note methodologique.** Rastrigin est fortement bruitee a NFE=5000 (cf [MGS-10](MGS-10-CenterBias.ipynb)) : on moyenne sur 5 seeds et on lit les resultats comme des **tendances**, pas des certitudes.

In [3]:
// === TUNING (fixe) === proba de mutation statique, identique chaque generation.
IMetaHeuristic BuildFixed(double p)
{ var m=new DefaultMetaHeuristic();
  m.ProbabilityConfig.Mutation.Strategy=ProbabilityStrategy.TestProbability|ProbabilityStrategy.OverwriteProbability;
  m.ProbabilityConfig.Mutation.StaticProbability=(float)p;
  return m; }

Console.WriteLine("Rastrigin (dim=5, NFE=5000, moy 5 seeds) -- objectif (proche 0 = meilleur)");
double bestFixed=double.PositiveInfinity; double bestP=0;
foreach (var p in new[]{0.01, 0.05, 0.2, 0.5})
{ double obj=RunAvg(g=>BuildFixed(p), reqR, Seeds);
  if (obj<bestFixed){bestFixed=obj; bestP=p;}
  Console.WriteLine($"  p={p,5:F2}   objectif {obj,8:F3}");
}
Console.WriteLine($"  -> meilleur reglage fixe : p={bestP:F2} (objectif {bestFixed:F3}).");
Console.WriteLine("     Aucun taux fixe n'est robuste : le 'bon' depend du paysage et du budget.");

Rastrigin (dim=5, NFE=5000, moy 5 seeds) -- objectif (proche 0 = meilleur)


  p= 0,01   objectif   12,378


  p= 0,05   objectif   11,885


  p= 0,20   objectif   12,332


  p= 0,50   objectif   12,753


  -> meilleur reglage fixe : p=0,05 (objectif 11,885).


     Aucun taux fixe n'est robuste : le 'bon' depend du paysage et du budget.


Le meilleur reglage fixe existe, mais il est **specifique a ce paysage et a ce budget**. C'est l'argument contre le reglage : un taux uniforme est forcement sous-optimal a une extremite de la course (trop timide pour explorer au debut, ou trop bruyant pour converger a la fin). Le **controle** adresse cela.

## 2. Controle deterministe — un schedule `f(generation)`

**Idee.** Au lieu de choisir `p`, on definit une **fonction decroissante** du numero de generation : mutation **haute au debut** (explorer largement), **basse a la fin** (exploiter le meilleur bassin trouve). C'est le controle **deterministe** (Eiben) : la variation ne depend d'aucun feedback, uniquement du temps.

Un schedule exponentiel : `p(g) = p_max * exp(-k * g/G)`, avec `p_max = 0.5`, `k = 3` -> `p(0)=0.5`, `p(G) ~= 0.025`.

Le fork expose ce mecanisme via **`ProbabilityConfig.Mutation.DynamicProbability : Func<IEvolutionContext, float, float>`** combine a `Strategy = OverwriteProbability`. A chaque evaluation de la probabilite, le moteur appelle notre fonction avec le contexte d'evolution (`ctx.Population.GenerationsNumber`).

In [4]:
// Visualisation SVG inline du schedule deterministe : p(g) = 0.5 * exp(-3 * g/G).
// Prong-A (#3801, #6927) : rendu SVG statique zero-dependance (rend sur GitHub/nbviewer/offline).
// Remplace le chart Plotly-CDN (C548-L2) dont le script externe rendait BLANC en consultation statique.
// Helper canon SvgChartHelper.cs (#6942 MERGED) ; primitive Overlay multi-series (#6958 MERGED).
#load "../../Probas/Infer/SvgChartHelper.cs"

// === CONTROL DETERMINISTE === schedule decroissant p(g) = 0.5 * exp(-3 * g/G)
IMetaHeuristic BuildDecay(int maxGen)
{ var m=new DefaultMetaHeuristic();
  m.ProbabilityConfig.Mutation.Strategy=ProbabilityStrategy.TestProbability|ProbabilityStrategy.OverwriteProbability;
  m.ProbabilityConfig.Mutation.DynamicProbability=(ctx, initial) => {
      int g=ctx.Population?.GenerationsNumber ?? 0;
      double t=(double)g/Math.Max(1,maxGen);
      return (float)(0.5*Math.Exp(-3.0*t));   // 0.5 -> ~0.025
  };
  return m; }

// Courbe continue du schedule de decroissance : p(g) = 0.5 * exp(-3 * g/G).
int G = NFE / PopSize;
double Decay(int g) => 0.5 * Math.Exp(-3.0 * (double)g / G);
var gAxis = Enumerable.Range(0, G + 1).Select(g => (double)g).ToArray();
var pCurve = gAxis.Select(g => Decay((int)g)).ToArray();

// 3 series sur axe X numerique partage (generation g) -> SvgChartHelper.Overlay (pattern canon #6927).
var series = new SvgSeries[]
{
    new SvgSeries("p(g) (probabilite de mutation)", gAxis, pCurve, TraceStyle.Line, "#4C72B0"),
    new SvgSeries("Exploration (debut, p=0.5)", new[] { 0.0 }, new[] { Decay(0) }, TraceStyle.Markers, "#DD8452"),
    new SvgSeries("Exploitation (fin, p~0.025)", new[] { (double)G }, new[] { Decay(G) }, TraceStyle.Markers, "#C44E52"),
};
display(SvgChartHelper.Overlay($"Schedule deterministe : p(g) = 0.5 * exp(-3 * g/G) (G={G})", "generation g", "p(g)", series));

Console.WriteLine($"Schedule deterministe p(g) = 0.5 * exp(-3 * g/G),  G=" + G);
Console.WriteLine($"  Exploration tot (p=0.5) -> Exploitation tard (p={Decay(G):F3}) : le mecanisme f(generation) est valide par la courbe.");

double decayObj=RunAvg(g=>BuildDecay(g), reqR, Seeds);
Console.WriteLine();
Console.WriteLine($"Rastrigin -- controle deterministe (decay) : objectif {decayObj:F3}");
Console.WriteLine($"  vs meilleur reglage fixe p={bestP:F2}      : objectif {bestFixed:F3}");
Console.WriteLine($"  => le deterministe {(decayObj < bestFixed ? "BAT" : "est competitif avec (a bruit pres)")} le meilleur fixe (delta {decayObj-bestFixed:+0.000;-0.000;0.000:F3}).");


Schedule deterministe : p(g) = 0.5 * exp(-3 * g/G) (G=100) -0.004 0.129 0.262 0.395 0.529 0 25 50 75 100 generation g p(g) <polyline points='52,55.643 59.52,66.862 67.04,77.75 74.56,88.316 82.08,98.569 89.6,108.52 97.12,118.177 104.64,127.548 112.16,136.642 119.68,145.467 127.2,154.032 134.72,162.343 142.24,170.409 149.76,178.237 157.28,185.833 164.8,193.204 172.32,200.358 179.84,207.3 187.36,214.038 194.88,220.576 202.4,226.92 209.92,233.078 217.44,239.053 224.96,244.852 232.48,250.479 240,255.94 247.52,261.24 255.04,266.383 262.56,271.374 270.08,276.217 277.6,280.917 285.12,285.479 292.64,289.905 300.16,294.201 307.68,298.37 315.2,302.416 322.72,306.342 330.24,310.152 337.76,313.849 345.28,317.437 352.8,320.919 360.32,324.299 367.84,327.578 375.36,330.76 382.88,333.849 390.4,336.846 397.92,339.754 405.44,342.577 412.96,345.316 420.48,347.974 428,350.554 435.52,353.057 443.04,355.486 450.56,357.844 458.08,360.132 465.6,362.352 473.12,364.507 480.64,366.598 488.16,368.627 495.68,370.596 503.2,372.507 510.72,374.362 518.24,376.161 525.76,377.908 533.28,379.603 540.8,381.248 548.32,382.844 555.84,384.393 563.36,385.896 570.88,387.355 578.4,388.771 585.92,390.145 593.44,391.478 600.96,392.772 608.48,394.027 616,395.246 623.52,396.428 631.04,397.576 638.56,398.69 646.08,399.77 653.6,400.819 661.12,401.837 668.64,402.825 676.16,403.783 683.68,404.713 691.2,405.616 698.72,406.492 706.24,407.342 713.76,408.167 721.28,408.968 728.8,409.745 736.32,410.499 743.84,411.231 751.36,411.941 758.88,412.63 766.4,413.298 773.92,413.947 781.44,414.577 788.96,415.188 796.48,415.782 804,416.357' fill='none' stroke='#4C72B0' stroke-width='2'/> p(g) (probabilite de mutation) Exploration (debut, p=0.5) Exploitation (fin, p~0.025)

Schedule deterministe p(g) = 0.5 * exp(-3 * g/G),  G=100


  Exploration tot (p=0.5) -> Exploitation tard (p=0,025) : le mecanisme f(generation) est valide par la courbe.


Rastrigin -- controle deterministe (decay) : objectif 12,332


  vs meilleur reglage fixe p=0,05      : objectif 11,885


  => le deterministe est competitif avec (a bruit pres) le meilleur fixe (delta +0,446).


Le decay est **competitif** avec le meilleur reglage fixe (a bruit pres, NFE=5000) — il ne le domine pas nettement. Ce n'est pas un echec du controle deterministe, mais un indice honnete : sur ce paysage et ce budget, un schedule simple n'apporte pas un avantage massif. Le controle qui dominera arrive au §4 (self-adaptatif). Ce que cette section etablit est surtout le **mecanisme** : `DynamicProbability = f(generation)` produit bien un taux continu decroissant (trace ci-dessus), et le moteur l'invoque reellement chaque generation.

## 3. Controle adaptatif — la diversite pilote le taux

**Idee.** Au lieu d'un schedule aveugle, on **observe la population** : si elle est **convergee** (faible diversite, tous concentres), on **remonte** la mutation pour forcer l'exploration. Si elle est **diverse**, on **baisse** la mutation pour laisser la convergence se faire. C'est le controle **adaptatif** : la variation est pilotee par un *feedback*.

C'est l'esprit de la celebre **regle 1/5 de Rechenberg** (1973) : on augmente le pas de mutation si plus d'un cinquieme des mutations ameliorent, on le diminue sinon. Ici on utilise un signal de diversite equivalent (ecart-type des genes), disponible dans le contexte d'evolution.

Le meme `DynamicProbability` lit `ctx.Population.CurrentGeneration.Chromosomes`. Pour ne pas le recalculer a chaque evaluation, on le **cache par generation** (fermeture capturant un etat mutable).

> **Honestete anticipée.** La valeur de l'adaptatif depend entierement de la **qualite du signal** de feedback. Notre signal (ecart-type du premier gene) est volontairement simple : il peut mal calibrer. C'est le defi central de l'adaptatif, pas un echec du concept.

In [5]:
// === CONTROL ADAPTATIF === diversite (ecart-type des genes) pilote le taux.
// faible diversite -> forte mutation (escape) ; forte diversite -> faible mutation (converge).
IMetaHeuristic BuildAdaptive((double lo,double hi) bounds)
{ var m=new DefaultMetaHeuristic();
  m.ProbabilityConfig.Mutation.Strategy=ProbabilityStrategy.TestProbability|ProbabilityStrategy.OverwriteProbability;
  double range=bounds.hi-bounds.lo;
  int cacheGen=-1; double cacheSd=range;   // etat de cache par generation (fermeture)
  m.ProbabilityConfig.Mutation.DynamicProbability=(ctx, initial) => {
      int g=ctx.Population?.GenerationsNumber ?? 0;
      if (g!=cacheGen)
      { cacheGen=g;
        var chroms=ctx.Population?.CurrentGeneration?.Chromosomes;
        var vals=(chroms??new System.Collections.Generic.List<IChromosome>())
                 .Select(c => { try { return ((DoubleArrayChromosome)c).GetDoubleValues()[0]; } catch { return double.NaN; } })
                 .Where(v => !double.IsNaN(v)).ToArray();
        cacheSd = vals.Length>=2 ? Math.Sqrt(vals.Average(v => { double d=v-vals.Average(); return d*d; })) : range;
      }
      double norm=Math.Min(1.0, cacheSd/(range*0.15));   // 1 = diverse, 0 = convergé
      return (float)(0.05 + 0.45*(1.0-norm));            // 0.05 (diverse) .. 0.50 (convergé)
  };
  return m; }

double adaptObj=RunAvg(g=>BuildAdaptive(KnownFunctionsBounds.For(typeof(RastriginFitness))), reqR, Seeds);
Console.WriteLine($"Rastrigin -- controle adaptatif (diversite) : objectif {adaptObj:F3}");
Console.WriteLine($"  vs deterministe (decay) {decayObj:F3}  | meilleur fixe {bestFixed:F3}");
Console.WriteLine("  Lecture : ici l'adaptatif SOUS-performe. Pourquoi ? Le signal de diversite (ecart-type");
Console.WriteLine("  du 1er gene) sous-mute au depart (population diverse -> p=0.05) puis sur-mute tard.");
Console.WriteLine("  => Leçon : la valeur de l'adaptatif tient a la QUALITE du signal (cf Exercice 2 : regle 1/5 par succes).");

Rastrigin -- controle adaptatif (diversite) : objectif 11,913


  vs deterministe (decay) 12,332  | meilleur fixe 11,885


  Lecture : ici l'adaptatif SOUS-performe. Pourquoi ? Le signal de diversite (ecart-type


  du 1er gene) sous-mute au depart (population diverse -> p=0.05) puis sur-mute tard.


  => Leçon : la valeur de l'adaptatif tient a la QUALITE du signal (cf Exercice 2 : regle 1/5 par succes).


## 4. Controle self-adaptatif — un taux par individu

**Idee.** Au lieu d'un taux unique pour toute la population, **chaque individu porte son propre taux** : certains explorent (taux eleve), d'autres exploitent (taux faible) — une population **heterogene**. C'est le controle **self-adaptatif** (Eiben) : le parametre est attache a l'individu.

Le contexte d'evolution expose **`ctx.OriginalIndex`** — l'indice de l'individu cible dans la population. On l'utilise pour assigner un taux qui varie par individu (une grille reguliere sur `[0.05, 0.5]`).

> **Honestete.** Le *vrai* self-adaptatif (Evolution Strategies, Schwefel) encode le taux **dans le chromosome** : le taux mute et est selectionne *avec* la solution. Notre demo montre le **mecanisme** (parametre par individu via le contexte) ; l'encodage chromosome est l'[**Exercice 3**](#Exercice-3-:-taux-encode-dans-le-chromosome-(vrai-self-adaptatif)).

In [6]:
// === CONTROL SELF-ADAPTATIF === un taux par individu via ctx.OriginalIndex.
// Population heterogene : taux reparti sur [0.05, 0.50] selon l'indice de l'individu.
IMetaHeuristic BuildSelfAdaptive(int popSize)
{ var m=new DefaultMetaHeuristic();
  m.ProbabilityConfig.Mutation.Strategy=ProbabilityStrategy.TestProbability|ProbabilityStrategy.OverwriteProbability;
  m.ProbabilityConfig.Mutation.DynamicProbability=(ctx, initial) => {
      int idx=ctx.OriginalIndex;               // -1 au niveau population -> taux moyen
      if (idx<0) return 0.2f;
      double frac=(double)(idx % popSize)/popSize;
      return (float)(0.05 + 0.45*frac);         // 0.05 (individu 0) .. ~0.50 (individu popSize-1)
  };
  return m; }

double selfObj=RunAvg(g=>BuildSelfAdaptive(PopSize), reqR, Seeds);
Console.WriteLine($"Rastrigin -- controle self-adaptatif (heterogene) : objectif {selfObj:F3}");
Console.WriteLine($"  vs adaptatif {adaptObj:F3}  | deterministe {decayObj:F3}  | meilleur fixe {bestFixed:F3}");
Console.WriteLine("  => la population heterogene HEDGE : a chaque instant, une sous-population a le bon");
Console.WriteLine("     taux pour la phase courante. EliteSelection preserve le meilleur des deux mondes.");

Rastrigin -- controle self-adaptatif (heterogene) : objectif 9,944


  vs adaptatif 11,913  | deterministe 12,332  | meilleur fixe 11,885


  => la population heterogene HEDGE : a chaque instant, une sous-population a le bon


     taux pour la phase courante. EliteSelection preserve le meilleur des deux mondes.


L'heterogeneite **hedger ses paris** : au lieu de parier sur un seul taux, la population en essaie plusieurs en parallele. C'est aussi le pont vers les **strategies d'evolution** (ES) et l'auto-adaptation ou le taux lui-meme evolue.

In [7]:
// === COMPARAISON === Rastrigin (multimodal) vs Sphere (unimodal) — le spoiler honnete se confirme.
double rFix  =bestFixed;
double rDecay=RunAvg(g=>BuildDecay(g), reqR, Seeds);
double rAdapt=RunAvg(g=>BuildAdaptive(KnownFunctionsBounds.For(typeof(RastriginFitness))), reqR, Seeds);
double rSelf =RunAvg(g=>BuildSelfAdaptive(PopSize), reqR, Seeds);

double sFix  =RunAvg(g=>BuildFixed(bestP), reqS, Seeds);
double sDecay=RunAvg(g=>BuildDecay(g), reqS, Seeds);
double sAdapt=RunAvg(g=>BuildAdaptive(KnownFunctionsBounds.For(typeof(SphereFitness))), reqS, Seeds);
double sSelf =RunAvg(g=>BuildSelfAdaptive(PopSize), reqS, Seeds);

Console.WriteLine("=====================  COMPARAISON (moy 5 seeds, objectif : proche 0 = meilleur)  =====================");
Console.WriteLine("Strategie                | Rastrigin (multimodal) | Sphere (unimodal)");
Console.WriteLine("-------------------------|------------------------|------------------");
Console.WriteLine($"Reglage fixe p={bestP:F2}        | {rFix,18:F3}     | {sFix,12:F3}");
Console.WriteLine($"Controle deterministe    | {rDecay,18:F3}     | {sDecay,12:F3}");
Console.WriteLine($"Controle adaptatif       | {rAdapt,18:F3}     | {sAdapt,12:F3}");
Console.WriteLine($"Controle self-adaptatif  | {rSelf,18:F3}     | {sSelf,12:F3}");
Console.WriteLine("------------------------------------------------------------------------------------------------------");
Console.WriteLine("Lecture (le spoiler honnete se confirme) :");
Console.WriteLine("  - Rastrigin : le self-adaptatif gagne NETTEMENT (hedging) ; fixe/decay/adaptatif sont dans le bruit (~12).");
Console.WriteLine("  - Sphere    : l'ordre s'inverse : decay et fixe (peu de mutation) menent ; self-adaptatif dernier.");
Console.WriteLine("  => Aucune strategie ne domine sur les deux paysages : No-Free-Lunch au niveau du PARAMETRE.");
Console.WriteLine("     La mutation = exploration : elle aide multimodal (extraire optima locaux) et nuit unimodal (disrupte la convergence).");
Console.WriteLine("     Le decay decroit vite vers un taux bas (~0.025) : profil proche d'un fixe faible -> bon pour Sphere,");
Console.WriteLine("     timide pour Rastrigin. Le self-adaptatif pousse l'exploration au max -> l'effet inverse.");

=====================  COMPARAISON (moy 5 seeds, objectif : proche 0 = meilleur)  =====================


Strategie                | Rastrigin (multimodal) | Sphere (unimodal)


-------------------------|------------------------|------------------


Reglage fixe p=0,05        |             11,885     |        0,750


Controle deterministe    |             12,332     |        0,711


...

  - Sphere    : l'ordre s'inverse : decay et fixe (peu de mutation) menent ; self-adaptatif dernier.


  => Aucune strategie ne domine sur les deux paysages : No-Free-Lunch au niveau du PARAMETRE.


     La mutation = exploration : elle aide multimodal (extraire optima locaux) et nuit unimodal (disrupte la convergence).


     Le decay decroit vite vers un taux bas (~0.025) : profil proche d'un fixe faible -> bon pour Sphere,


     timide pour Rastrigin. Le self-adaptatif pousse l'exploration au max -> l'effet inverse.


## Synthese — quand utiliser quelle strategie ?

| Strategie | Coute (en tuning) ? | Adapte au paysage ? | Adapte au cours du temps ? | Quand l'utiliser |
|-----------|:---:|:---:|:---:|-----------------|
| **Reglage** (fixe) | faible (1 essai) | non | non | Paysage facile/unimodal connu, baseline |
| **Controle deterministe** (schedule) | moyen (forme du schedule) | non | **oui** (temps) | Budget connu, mecanisme simple a calibrer |
| **Controle adaptatif** (feedback) | moyen (signal) | **oui** | **oui** (etat) | Budget inconnu, stagnation — *si* bon signal |
| **Controle self-adaptatif** (par individu) | eleve (espace du taux) | **oui** | **oui** (individu) | Paysage multimodal dur, hedging |

**La leçon centrale (No-Free-Lunch du parametre).** Meme avec le controle, **aucune strategie ne domine universellement** : le self-adaptatif gagne nettement sur Rastrigin (multimodal) mais perd sur Sphere (unimodal) ; le decay et le fixe (peu de mutation) font l'inverse. La **mutation est le levier d'exploration** — precieux quand le paysage regorge d'optima locaux (il faut s'extraire), nuisible quand il est convexe (il faut converger vite). Le paysage (cf [MGS-15](MGS-15-LandscapeAnalysis.ipynb)) dicte donc, la encore, le choix.

C'est la **deuxieme reponse au No-Free-Lunch**, en miroir de la [selection d'algorithme (MGS-16)](MGS-16-AlgorithmSelection.ipynb) :

> *Pas de bon algorithme universel -> on **selectionne** l'algorithme (MGS-16) et on **controle** ses parametres (MGS-17). Mais le controle non plus n'est pas universel : le paysage reste souverain.*

## Exercice 1 : schedule lineaire vs exponentiel

Le schedule deterministe utilise ici une decroissance **exponentielle**. Implementez une decroissance **lineaire** `p(g) = p_max * (1 - g/G)` et comparez. Lequel convient mieux a Rastrigin ?

In [8]:
// Exercice 1 : schedule LINEAIRE p(g) = 0.5 * (1 - g/G)
// Indice : reprenez BuildDecay, remplacez Math.Exp(-3*t) par (1-t).
// Etape 1 : ecrire BuildLinear(int maxGen) avec DynamicProbability = (ctx,p) => (float)(0.5*(1 - g/G)).
// Etape 2 : lancer RunAvg(g=>BuildLinear(g), reqR, Seeds) et comparer a decayObj.

// TODO etudiant : implementer BuildLinear et afficher l'objectif.
double linearObj = 0.0; // TODO : remplacer par RunAvg(g=>BuildLinear(g), reqR, Seeds)
Console.WriteLine($"Exercice a completer : schedule lineaire (cf decay exponentiel = {decayObj:F3})");

Exercice a completer : schedule lineaire (cf decay exponentiel = 12,332)


## Exercice 2 : la regle 1/5 de Rechenberg (adaptatif par succes)

Le controle adaptatif ci-dessus utilise la **diversite** comme signal — et sous-performe a cause d'un signal trop crude. La regle **1/5** (Rechenberg, 1973) utilise le **taux de succes** : la fraction des mutations qui ameliorent la fitness parent.
- si > 1/5 : le pas est trop petit -> **augmenter** la mutation ;
- si < 1/5 : le pas est trop grand -> **diminuer**.

Indice : il faut un etat mutable (fermeture) qui memorise le meilleur fitness de la generation precedente, et compare.

In [9]:
// Exercice 2 : regle 1/5 de Rechenberg
// Indice : fermeture capturant (int derniereGen, double meilleurFitness, float tauxCourant).
// Etape 1 : a chaque changement de generation, calculer le taux de succes (nb ameliorations / pop).
// Etape 2 : ajuster tauxCourant *= 1.1 si succes>0.2, *= 0.9 sinon (clamper [0.01, 0.6]).

// TODO etudiant : implementer BuildOneFifth() et comparer a l'adaptatif-diversite.
double oneFifthObj = 0.0; // TODO : remplacer par RunAvg(g=>BuildOneFifth(), reqR, Seeds)
Console.WriteLine($"Exercice a completer : regle 1/5 (cf adaptatif-diversite = {adaptObj:F3})");

Exercice a completer : regle 1/5 (cf adaptatif-diversite = 11,913)


## Exercice 3 : taux encode dans le chromosome (vrai self-adaptatif)

La demo du §4 assigne un taux par **indice**. Le *vrai* self-adaptatif encode le taux **dans le chromosome** : le taux mute et est selectionne avec la solution.

Indice : etendez `DoubleArrayChromosome` avec un gene supplementaire `sigma` (le taux), mutez-le, et lisez-le dans `DynamicProbability`.

In [10]:
// Exercice 3 : chromosome a taux encode (Evolution Strategies)
// Indice : public class SelfAdaptiveChromosome : ChromosomeBase { double[] genes; double sigma; ... }
// Etape 1 : CreateNew tire sigma dans [0.01, 0.5] ; GenerateGene mute sigma aussi.
// Etape 2 : DynamicProbability lit le sigma du chromosome cible (cast dans BuildSelfAdaptive).
// Etape 3 : le fitness n'evalue que les `genes` (sigma est un parametre, pas une dimension).

// TODO etudiant : definir SelfAdaptiveChromosome et BuildSelfAdaptiveEncoded().
Console.WriteLine("Exercice a completer : taux encode dans le chromosome (vrai self-adaptatif ES).");

Exercice a completer : taux encode dans le chromosome (vrai self-adaptatif ES).


## Exercice 4 : controler le CROSSOVER au lieu de la mutation

Toutes les demos controlent la **mutation** (le levier d'exploration). Le meme mecanisme s'applique au **crossover** : `ProbabilityConfig.Crossover.DynamicProbability`. Par exemple, un crossover faible en debut (laisser la mutation explorer) puis fort en fin (exploiter par recombinaison).

In [11]:
// Exercice 4 : controle deterministe du CROSSOVER
// Indice : m.ProbabilityConfig.Crossover.Strategy = OverwriteProbability ;
//          m.ProbabilityConfig.Crossover.DynamicProbability = (ctx,p) => schedule(g) ;
// Attention : pour isoler l'effet, fixez aussi la mutation (BuildFixed) a un niveau bas.
// Question : sur Rastrigin, le crossover-control aide-t-il autant que le mutation-control ?

// TODO etudiant : implementer BuildCrossoverDecay() et comparer.
Console.WriteLine($"Exercice a completer : controle du crossover (cf mutation-control = {decayObj:F3}).");

Exercice a completer : controle du crossover (cf mutation-control = 12,332).


## Conclusion

| Concept | Ce notebook |
|---------|-------------|
| **Taxonomie d'Eiben** | reglage (fixe) vs controle (deterministe / adaptatif / self-adaptatif) |
| **Mecanisme du moteur** | `ProbabilityConfig.{Mutation,Crossover}.DynamicProbability : Func<IEvolutionContext,float,float>` + `OverwriteProbability` |
| **Deterministe** | schedule `f(generation)` — explore tot, exploite tard — competitif, mecanisme simple |
| **Adaptatif** | feedback de diversite — s'adapte, **mais sensible a la qualite du signal** |
| **Self-adaptatif** | taux par individu via `ctx.OriginalIndex` — **hedging**, gagne sur multimodal |
| **Leçon centrale** | **No-Free-Launch du parametre** : aucune strategie ne domine Rastrigin ET Sphere |

**Le controle de parametres retire le pari du reglage fixe** — mais n'offre pas de garantie universelle. La mutation etant le levier d'**exploration**, le controle aide quand l'exploration a de la valeur (paysage multimodal) et nuit quand elle n'en a pas (paysage unimodal). Le **self-adaptatif** emerge comme le champion du multimodal (hedging : la population essaie plusieurs taux en parallele) ; le **decay** et le **fixe** restent competitifs sans dominer. Combine a la [selection d'algorithme (MGS-16)](MGS-16-AlgorithmSelection.ipynb), le controle forme la reponse pragmatique au **No-Free-Lunch** : on choisit le bon moteur **et** on pilote ses reglages selon le paysage (decrit en [MGS-15](MGS-15-LandscapeAnalysis.ipynb)).

### Limites honnetes
- **Aucune strategie n'est universelle** : le self-adaptatif (champion Rastrigin) est le pire sur Sphere. C'est le No-Free-Lunch, pas un defaut de implementation.
- L'**adaptatif** depend de la **qualite du signal** de feedback ; un signal pauvre (notre ecart-type du 1er gene) le pousse a mal calibrer. Une regle 1/5 par succes (Exercice 2) est un meilleur signal.
- Le *vrai* self-adaptatif (taux evolue) coute une **dimension supplementaire** dans l'espace de recherche (Exercice 3).
- A NFE=5000, Rastrigin reste **bruite** (cf MGS-10/16) : les comparaisons sont moyennées sur 5 seeds et sont des **tendances**, pas des certitudes ; les ecarts types sont de l'ordre de 1-2 unites.

### Pour aller plus loin
- **Eiben, Hinterding, Michalewicz (1999)**, *Parameter Control in Evolutionary Algorithms* — la taxonomie canonique.
- **Karafotias, Hoogendoorn, Eiben (2015)**, *Parameter Control in Evolutionary Algorithms: A Tutorial*.
- **Rechenberg (1973)**, *Evolutionsstrategie* — la regle 1/5.
- **Schwefel (1995)**, *Evolution and Optimum Seeking* — self-adaptation en ES.
- Ponts : [MGS-16 Selection d'algorithme](MGS-16-AlgorithmSelection.ipynb) | [MGS-15 Analyse de paysage](MGS-15-LandscapeAnalysis.ipynb) | [MGS-10 Biais central](MGS-10-CenterBias.ipynb)